# Load processed data + merged temperature

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sklearn.feature_selection import mutual_info_regression

from energy_forecast.features.calendar import create_calendar_features, add_cyclical_encoding
from energy_forecast.features.lags import create_day_ahead_features

In [ ]:
if os.getcwd().endswith("notebooks"):
    os.chdir("..")
df = pd.read_csv("data/processed/demand_with_temperature.csv", index_col=0, parse_dates=True)
# Saved in UTC; calendar features need local (Europe/London) time
df.index = df.index.tz_convert("Europe/London")
df

# Correlation Check — Kendall vs. Mutual Information

### Kendall

In [ ]:
df_diff = df.diff()

# Kendall on levels picks up shared long-term trends; on differences it measures short-term
# co-movement, which is what mutual information (below) is computed on as well
kendall = pd.DataFrame({
    "levels": df.corr(method="kendall", numeric_only=True)["nd"],
    "differences": df_diff.corr(method="kendall", numeric_only=True)["nd"],
}).drop(index="nd")
kendall.sort_values("differences", key=abs, ascending=False)

### Mutual information

In [ ]:
X = df_diff.drop(columns=['nd']).dropna()
y = df_diff.loc[X.index, 'nd']

mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
mi_series

Kendall correlation is computed twice: on **levels**, where shared long-term trends dominate,
and on **differences**, which measures short-term co-movement. Mutual information is computed on
differences, so the like-for-like comparison of the two methods is Kendall (differences) vs
mutual information; the levels column shows which relationships are mostly trend.

*The values quoted below are from an earlier run; re-check them against the tables above after
re-running.*

Mutual information ranks `embedded_solar_generation` as the strongest relevant feature
(0.219), while Kendall on levels almost entirely missed it (0.067) — consistent with solar
generation's relationship to demand being confounded with time-of-day rather than monotonic.
This directly supports the decision to retain solar and wind generation as features, in
contrast to the original thesis's approach of dropping them based on a limited OLS test.

Conversely, `embedded_solar_capacity` and `embedded_wind_capacity`, which showed the strongest
Kendall correlations on levels, drop to near-zero mutual information — confirming these
correlations were likely a trend artifact rather than a genuine short-term relationship, since
capacity changes infrequently rather than daily. Kendall on differences should show the same
near-zero relationship.

`temperature` (0.085) and `pump_storage_pumping` (0.157) show consistent, moderate relevance
across both methods and are retained with confidence.

`settlement_period` shows low mutual information despite known time-of-day effects on demand
(established via the boxplots and MSTL decomposition earlier). This is likely a limitation of
applying mutual information to differenced cyclical data rather than evidence that time-of-day
is unimportant; time-of-day will instead be captured via cyclical (sine/cosine) encodings in the
feature engineering step below, rather than relying on this correlation check alone.

**Features retained going forward:** temperature, pump_storage_pumping, non_bm_stor,
embedded_solar_generation, embedded_wind_generation, is_holiday, and all six interconnector
flow columns (`ifa_flow`, `ifa2_flow`, `britned_flow`, `moyle_flow`, `east_west_flow`,
`nemo_flow`; weak individually but potentially useful in combination, to be confirmed via SHAP).
`embedded_solar_capacity` and `embedded_wind_capacity` are dropped, as their correlation appears
to be a trend artifact rather than genuine signal.

In [ ]:
df = df.drop(columns=["embedded_solar_capacity", "embedded_wind_capacity"])

# Feature Construction

With the correlation analysis complete, the following cells build the actual
feature set to be used for modelling. Calendar, cyclical, lag, and rolling
features are all implemented as reusable functions in
`src/energy_forecast/features/`, rather than written inline, so they can be
reused consistently across notebooks and in the final training pipeline.

- **Calendar features** (`create_calendar_features`) provide the basic date/time
  components (hour, day of week, month, etc.) established as relevant in the
  earlier EDA (via boxplots and MSTL seasonality). `year` is created but left out
  of the saved feature set: the test period (2021–2024) lies beyond the training
  years, and models can't extrapolate it reliably (trees treat every later year as
  the last one seen). `year` would likely look important when fitting the training
  data, since GB demand fell steadily over 2009–2021 and `year` is a clean proxy for
  that trend, but that reliance does not carry over to unseen later years. The
  current demand level is instead captured by the lag and rolling features, which
  adapt automatically when demand shifts (e.g. the 2022 energy-crisis drop).
- **Cyclical encoding** (`add_cyclical_encoding`) converts the cyclical calendar
  columns (hour, day of week, month) into sine/cosine pairs, avoiding the
  artificial discontinuity of raw integer encoding at wraparound points (e.g.
  hour 23 to hour 0).
All demand-based and grid features are built by `create_day_ahead_features` in
`src/energy_forecast/features/lags.py`, which is unit-tested to use only information at least
48 steps old.

- **Lag features** (`create_lag_features`) are built at lags 48 and 336. The
  ACF/PACF analysis in the previous notebook identified lags 1, 2, 48 and 336 as
  carrying independent signal, but this project forecasts **day-ahead**
  (horizon h = 48 half-hours, direct strategy): to predict demand at time t, only
  demand observed up to t − 48 may be used. Lags 1 and 2 would not be available
  a day in advance, so only the lags ≥ 48 are kept.
- **Rolling statistics** (`create_rolling_features`) add 1-day and 1-week rolling
  mean and standard deviation of demand, to capture recent level and volatility
  beyond a single lag point. The windows end 48 steps before the target
  (`shift=HORIZON`), for the same reason.

- **Grid-operation variables** (`non_bm_stor`, `pump_storage_pumping` and the six
  interconnector flows) are outcomes of the target half-hour itself, partly driven
  by that half-hour's demand, so they would not be known a day ahead. They are
  replaced by their values 48 steps earlier (e.g. `ifa2_flow_lag_48`), the same
  rule as for the demand lags.

Weather and embedded generation (temperature, embedded wind/solar) are kept as
observed values for the target half-hour. At day-ahead they would in practice be
forecasts, so the models are evaluated under the common "perfect weather forecast"
assumption.

In [ ]:
df = create_calendar_features(df)
df = add_cyclical_encoding(df, "hour", period=24)
df = add_cyclical_encoding(df, "day_of_week", period=7)
df = add_cyclical_encoding(df, "month", period=12)

# Day-ahead direct forecasting: only information observed up to t - HORIZON is used to predict t.
# Grid-operation variables are outcomes of the target half-hour itself and would not be known a
# day ahead, so they are replaced by their values HORIZON steps earlier (e.g. ifa2_flow_lag_48)
HORIZON = 48
GRID_COLUMNS = [
    "non_bm_stor", "pump_storage_pumping",
    "ifa_flow", "ifa2_flow", "britned_flow", "moyle_flow", "east_west_flow", "nemo_flow",
]
df = create_day_ahead_features(
    df, horizon=HORIZON, lags=[48, 336], windows=[48, 336], lagged_columns=GRID_COLUMNS
)

In [ ]:
df

## Handling Missing Values from Lag/Rolling Features

The lag and rolling features introduce missing values at the start of the series,
where insufficient history exists to compute them (up to 384 steps, i.e. the one-week
window plus the 48-step day-ahead shift). These rows are dropped, representing a negligible loss
given the dataset spans approximately 16 years.

In [ ]:
df.isna().sum()

In [ ]:
df = df.dropna()

In [ ]:
# Full candidate feature set (everything except the target and non-numeric columns).
# `year` is also excluded: the test years lie beyond the training range, where tree models
# can't extrapolate it and MLR/LSTM would extrapolate it unreliably. The demand level is
# already tracked by the lag and rolling features.
PROCESSED_DIR = Path("data/processed")

full_features = [col for col in df.columns if col not in ['nd', 'day_name', 'year']]
df_full = df[full_features + ['nd']]
# Saved in UTC so the timestamps read back cleanly (local time would mix +00:00 and +01:00)
df_full.tz_convert("UTC").to_csv(PROCESSED_DIR / "model_ready_features_day_ahead.csv")

In [ ]:
df_full.shape

## Final Feature Set for This Notebook

The full engineered feature set has been saved to
`data/processed/model_ready_features_day_ahead.csv`.

It includes all engineered features (calendar, cyclical encodings, lags, rolling statistics, and retained correlation-check columns), restricted to information available
day-ahead (48 steps before each target).
This is used to train both the XGBoost and LSTM models, and to compute SHAP importance from the trained XGBoost model, since SHAP requires visibility into every candidate feature to assess which ones genuinely matter.

## Next Steps

Feature selection by MRFO is not performed in this notebook. In the next
notebook, MRFO will be run twice — once using XGBoost as the fitness function,
once using LSTM — with matched population size and iteration count, so any
difference in the resulting feature subsets can be attributed to the model
rather than to an uneven search budget. Each model will then be trained on its
own matching MRFO selection (XGBoost on the XGBoost-selected features, LSTM on
the LSTM-selected features), alongside both models trained on the full feature
set above.